## 3. The Arnoldi relation: a smaller least-squares problem (`gmres_v2`)

*Same math, computed better.* Like Step 2, this step changes how GMRES computes its answer, not what the answer is. In exact arithmetic the new solver produces exactly the same iterates as `gmres_v1`. In floating point they should agree up to round-off, and we'll check that they do.

**Where Step 2 left off.** We want to solve $Ax = b$, where $A$ is a nonsingular $n \times n$ matrix that we can only apply to vectors, through a function `matvec` $: v \mapsto Av$. Starting from $x_0 = 0$, GMRES picks at step $k$ the vector $x_k$ in the Krylov space

$$
\mathcal{K}_k = \operatorname{span}\{\, b,\ Ab,\ \dots,\ A^{k-1} b \,\}
$$

whose residual $r_k = b - A x_k$ has the smallest 2-norm, and stops once $\|r_k\|_2 \le \text{rtol} \cdot \|b\|_2$.

Step 2 built an orthonormal basis $v_1, v_2, \dots$ of $\mathcal{K}_k$ with the **Arnoldi process**. It starts from $v_1 = b / \beta$, where $\beta = \|b\|_2$. Step $j$ takes one matvec, $w = A v_j$. It computes the components $h_{ij} = v_i^\mathsf{T} w$ of $w$ along $v_1, \dots, v_j$ and subtracts them all, then repeats the subtraction once on what is left, adding the second pass's components to the $h_{ij}$. That is classical Gram–Schmidt applied twice (Step 2, section 2.4), and it keeps the basis orthonormal to round-off. Then it normalises what is left: $h_{j+1,j} = \|w\|_2$ and $v_{j+1} = w / h_{j+1,j}$. Column $j$ of the $(k+1) \times k$ matrix $\bar H_k$ holds the numbers step $j$ computed, $h_{1j}, \dots, h_{j+1,j}$, with zeros below.

`gmres_v1` writes $x_k = V_k\, y$, where $V_k = [\, v_1 \mid \cdots \mid v_k \,]$ and $y \in \mathbb{R}^k$ is a weight vector, and solves

$$
\min_{y \in \mathbb{R}^k} \big\| b - (A V_k)\, y \big\|_2
$$

from scratch at every step. That least-squares problem has $n$ equations, it needs all of $A V_k$ in memory, and it doesn't use $\bar H_k$ at all.

**The plan for this step.** The $h_{ij}$ record what $A$ does to the basis vectors. That turns the problem above into one with $k+1$ equations and $k$ unknowns, in which $n$ doesn't appear. `gmres_v2` solves that small problem instead.

### Setup

[tutorial_utils.py](tutorial_utils.py) holds what every step of the tutorial uses: `CountingMatvec` from Step 1, the example problems and the plot style. Each example builder returns a pair `(A, b)`. As before, a solver only ever gets the matrix wrapped in a `CountingMatvec`; we use `A` directly only to *check* answers.

The example problems in this notebook, each a matrix $A$ with a right-hand side $b$:

- **`geo3`**: the 3×3 running example from Steps 1 and 2,
  $$
  A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & 1 \\ 1 & 0 & 2 \end{pmatrix}, \qquad b = (1, 1, 1) .
  $$
  Nonsymmetric and well conditioned, and small enough to check every number by hand.
- **`diag_narrow`**: the 100×100 diagonal matrix $A = \operatorname{diag}(\lambda_1, \dots, \lambda_{100})$ with eigenvalues evenly spaced from 1 to 1.5, $\lambda_i = 1 + 0.5\,(i-1)/99$, and $b = (1, \dots, 1)$. `gmres_v0` solved it in 11 steps.
- **`diag_spread`**: the 20×20 diagonal matrix $A = \operatorname{diag}(\lambda_1, \dots, \lambda_{20})$ with eigenvalues evenly spaced on a log scale from 1 to $10^4$, $\lambda_i = 10^{4(i-1)/19}$, and $b = (1, \dots, 1)$. `gmres_v0` failed on it.
- **`diag_spread(20, decades=12)`**: the same kind of 20×20 diagonal matrix with the eigenvalues spread over twelve decades, $\lambda_i = 10^{12(i-1)/19}$ from 1 to $10^{12}$, and $b = (1, \dots, 1)$, so $\operatorname{cond}(A) = 10^{12}$. Step 2 (section 2.4) used it to show one pass of Gram–Schmidt losing orthogonality. The end of section 3.3 uses it to show what that does to this step's small problem.
- **`convdiff_n`** (new in this step): the $n \times n$ tridiagonal matrix you get by discretising the 1D convection–diffusion equation $-u''(s) + c\, u'(s) = 1$ on $[0, 1]$, with $u(0) = u(1) = 0$, on $n$ grid points. Here $b = (1, \dots, 1)$. The velocity $c$ decides the symmetry: $c = 0$ gives a symmetric matrix, and $c \ne 0$ a nonsymmetric one. Section 3.3 derives its entries.

`%autoreload 2` picks up edits to `tutorial_utils.py` without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np  # noqa: I001
import matplotlib.pyplot as plt

from tutorial_utils import (
    # the matvec wrapper from Step 1
    CountingMatvec,
    # loading definitions from an earlier step's notebook
    import_from_notebook,
    # example problems; each returns (A, b)
    geo3, diag_narrow, diag_spread, convdiff_n,
    # checking one solver version against another
    compare_versions,
    # plot style
    style_axes, plot_residuals, plot_matrix, add_figure_caption, REFERENCE_COLOR, SERIES_COLORS,
)

np.set_printoptions(precision=4, suppress=True)

A_geo3, b_geo3 = geo3()              # 3 x 3, the running example from Steps 1 and 2
A_narrow, b_narrow = diag_narrow()   # 100 x 100 diagonal, eigenvalues in [1, 1.5]
A_spread, b_spread = diag_spread()   # 20 x 20 diagonal, eigenvalues from 1 to 1e4
A_spread_12, b_spread_12 = diag_spread(20, decades=12)   # 20 x 20 diagonal, eigenvalues from 1 to 1e12

**From Steps 1 and 2.** From Step 2 we need `gmres_v1`, the solver this step improves, the Arnoldi step `arnoldi_step_classical_gram_schmidt_twice`, and `arnoldi`, which runs $k$ steps and returns $V_{k+1} = [\, v_1 \mid \cdots \mid v_{k+1} \,]$, $\bar H_k$ and $\beta$. Step 2's two one-pass variants, `arnoldi_step_classical_gram_schmidt` and `arnoldi_step_modified_gram_schmidt`, appear only in the demonstration at the end of section 3.3. From Step 1 we need `krylov_matrix`, which builds the literal basis $K_k = [\, b \mid Ab \mid \cdots \mid A^{k-1} b \,]$, for one comparison. Instead of copying them, `import_from_notebook` loads them straight from [GMRES_2_arnoldi_orthogonalise.ipynb](GMRES_2_arnoldi_orthogonalise.ipynb) and [GMRES_1_krylov.ipynb](GMRES_1_krylov.ipynb). It runs only those notebooks' imports and function definitions, not their plots or examples. To read any of them here, run `arnoldi_step_classical_gram_schmidt_twice??` and so on.

In [ ]:
# Built in Steps 1 and 2. Loaded from those notebooks, so each stays the only copy.
(gmres_v1, arnoldi, arnoldi_step_classical_gram_schmidt_twice,
 arnoldi_step_classical_gram_schmidt, arnoldi_step_modified_gram_schmidt) = import_from_notebook(
    "GMRES_2_arnoldi_orthogonalise.ipynb", "gmres_v1", "arnoldi", "arnoldi_step_classical_gram_schmidt_twice",
    # the two one-pass variants, only for the demonstration at the end of 3.3
    "arnoldi_step_classical_gram_schmidt", "arnoldi_step_modified_gram_schmidt")
krylov_matrix = import_from_notebook("GMRES_1_krylov.ipynb", "krylov_matrix")

### 3.1 What the $h_{ij}$ record: the Arnoldi relation

Start with the first Arnoldi step on `geo3`, which Step 2 did by hand:

$$
A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & 1 \\ 1 & 0 & 2 \end{pmatrix}, \qquad b = (1, 1, 1) .
$$

It gave $\beta = \sqrt 3$, $v_1 = (1, 1, 1) / \sqrt 3$, $h_{11} = 10/3$, $h_{21} = \sqrt 2 / 3$ and $v_2 = (2, -1, -1) / \sqrt 6$. The step computed the remainder $w = A v_1 - h_{11} v_1$ and then set $v_2 = w / h_{21}$. Rearranged, that says

$$
A v_1 = h_{11} v_1 + h_{21} v_2 :
$$

the matvec, written in the new basis. Multiplying by $\beta$ gives $A b = \beta\, (h_{11} v_1 + h_{21} v_2)$, which should be Step 1's $Ab = (4, 3, 3)$.

In [ ]:
V, H, beta = arnoldi(CountingMatvec(A_geo3), b_geo3, 2)  # V_3, H-bar_2 and beta
v1, v2 = V[:, 0], V[:, 1]
h11, h21 = H[0, 0], H[1, 0]

print(f"h11 = {h11:.4f}  (10/3 = {10 / 3:.4f})    h21 = {h21:.4f}  (sqrt(2)/3 = {np.sqrt(2) / 3:.4f})")
print("beta * (h11 v1 + h21 v2) =", beta * (h11 * v1 + h21 * v2), "   (Step 1: A b = [4. 3. 3.])")

Every Arnoldi step can be rearranged the same way. The remainder $w$ was $A v_j$ minus its components along $v_1, \dots, v_j$, and it ended up as $h_{j+1,j}\, v_{j+1}$:

$$
A v_j - \sum_{i=1}^{j} h_{ij}\, v_i = h_{j+1,j}\, v_{j+1}
\qquad\Longrightarrow\qquad
A v_j = \sum_{i=1}^{j+1} h_{ij}\, v_i .
$$

With classical Gram–Schmidt applied twice, the stored $h_{ij}$ is the first pass's component plus the second's. The second is zero in exact arithmetic (Step 2, section 2.3), so $h_{ij} = v_i^\mathsf{T} A v_j$. In floating point the stored $h_{ij}$ is exactly what was subtracted from $A v_j$ in total, over both passes, which is why the relation below holds to round-off whatever the passes did.

The coefficients in the equation for $A v_j$ are exactly column $j$ of $\bar H_k$. So the $k$ equations above, side by side, are one matrix equation, the **Arnoldi relation**:

$$
A V_k = V_{k+1}\, \bar H_k ,
\qquad
V_{k+1} = [\, v_1 \mid \cdots \mid v_{k+1} \,] \in \mathbb{R}^{n \times (k+1)},
\qquad
\bar H_k \in \mathbb{R}^{(k+1) \times k} .
$$

Column $j$ of the left side is $A v_j$. Column $j$ of the right side is $V_{k+1}$ times column $j$ of $\bar H_k$, which is $\sum_i h_{ij} v_i$.

$h_{ij} = 0$ whenever $i > j + 1$, because $A v_j$ needs only $v_1, \dots, v_{j+1}$. A matrix that is zero below its first subdiagonal is called **upper Hessenberg**. For `geo3` at $k = 2$:

$$
\bar H_2 =
\begin{pmatrix} h_{11} & h_{12} \\ h_{21} & h_{22} \\ 0 & h_{32} \end{pmatrix}
=
\begin{pmatrix} 3.3333 & 0.4714 \\ 0.4714 & 2.1667 \\ 0 & 0.8660 \end{pmatrix} .
$$

In [ ]:
H_bar_2 = H        # (k + 1) x k = 3 x 2
V_2 = V[:, :2]     # n x k
V_3 = V            # n x (k + 1)
print("H-bar_2 =\n", H_bar_2)
print("\n||A V_2 - V_3 H-bar_2|| =", np.linalg.norm(A_geo3 @ V_2 - V_3 @ H_bar_2))  # checking only

Step 2 noted that `geo3`'s $\bar H_3$ has a second entry that prints as 0, $h_{13} = 1.9 \times 10^{-16}$, above the subdiagonal, where a Hessenberg matrix can have anything. In exact arithmetic it is exactly 0, but that is an accident of this $A$ and $b$, not a pattern. By the argument above, $h_{13} = v_1^\mathsf{T} A v_3 = (A^\mathsf{T} v_1)^\mathsf{T} v_3$. For this matrix $A^\mathsf{T} b = (4, 3, 3)$ (the column sums of $A$), and that happens to be perpendicular to $v_3 = (0, -1, 1)/\sqrt 2$. With a different $b$ the entry is not zero:

In [ ]:
_, H_ones, _ = arnoldi(CountingMatvec(A_geo3), b_geo3, 3)
_, H_other, _ = arnoldi(CountingMatvec(A_geo3), np.array([1.0, 2.0, 3.0]), 3)
print(f"b = (1, 1, 1):  h_13 = {H_ones[0, 2]:.1e}")
print(f"b = (1, 2, 3):  h_13 = {H_other[0, 2]:.4f}")

So `geo3` happens to hide part of the Hessenberg shape. Section 3.3 shows what $\bar H$ looks like in general.

### 3.2 Shrinking the problem: `gmres_v2`

The Arnoldi relation turns the $n \times k$ least-squares problem into a small one. It takes two facts.

1. **$b$ in the new basis.** $b = \beta v_1 = V_{k+1}\, (\beta e_1)$, where $e_1 = (1, 0, \dots, 0) \in \mathbb{R}^{k+1}$ is the first unit vector.
2. **$V_{k+1}$ preserves lengths.** For any $z \in \mathbb{R}^{k+1}$, $\|V_{k+1} z\|_2^2 = z^\mathsf{T} V_{k+1}^\mathsf{T} V_{k+1}\, z = z^\mathsf{T} z = \|z\|_2^2$, since $V_{k+1}^\mathsf{T} V_{k+1} = I$.

So for any $y \in \mathbb{R}^k$,

$$
b - A V_k\, y = V_{k+1} (\beta e_1) - V_{k+1} \bar H_k\, y = V_{k+1} \big( \beta e_1 - \bar H_k\, y \big)
\qquad\Longrightarrow\qquad
\|b - A V_k\, y\|_2 = \|\beta e_1 - \bar H_k\, y\|_2 .
$$

The GMRES problem is therefore

$$
\min_{y \in \mathbb{R}^k} \big\| \beta e_1 - \bar H_k\, y \big\|_2 ,
$$

a least-squares problem with $k + 1$ equations and $k$ unknowns. **$n$ doesn't appear in it at all.** With $n$ in the millions and $k$ in the tens, that is the difference between working with $n \times k$ arrays and with a matrix that fits on a screen. It also means $A V_k$ no longer has to be stored, which halves `gmres_v1`'s memory.

One caveat: fact 2 needs $V_{k+1}$ to be orthonormal. In floating point it is orthonormal only to round-off, so the two residuals agree only to about $\|I - V_{k+1}^\mathsf{T} V_{k+1}\|_2$. Step 2's Figure 2.1 measured it: with classical Gram–Schmidt applied twice it stays at about $4 \times 10^{-16}$. The end of section 3.3 shows what happens to this problem when it doesn't.

Check it on `geo3` at $k = 2$ with $\bar H_2$ from 3.1. (For a 3×3 matrix at $k = 2$, $n = k + 1$, so the two problems happen to be the same size.)

In [ ]:
beta_e1 = np.array([beta, 0.0, 0.0])                          # beta e_1, length k + 1 = 3
y_small, *_ = np.linalg.lstsq(H_bar_2, beta_e1, rcond=None)     # min ||beta e_1 - H-bar_2 y||
y_big, *_ = np.linalg.lstsq(A_geo3 @ V_2, b_geo3, rcond=None)   # min ||b - A V_2 y||  (A used for checking only)

print("y from the small problem:     ", y_small)
print("y from min ||b - A V_2 y||:   ", y_big)
print("||beta e_1 - H-bar_2 y|| =", np.linalg.norm(beta_e1 - H_bar_2 @ y_small), "   (Step 1: ||r_2|| = 0.0933)")
print("x_2 = V_2 y =", V_2 @ y_small, "   (Step 1: x_2 = [0.2261 0.3478 0.3478])")

Both problems give the same $y$, the same residual 0.0933 as in Step 1, and the same $x_2$.

**The algorithm, `gmres_v2`:**

1. Allocate `V` ($n \times (m+1)$) and `H` ($(m+1) \times m$) as zeros, with $m$ = `max_iterations`, and set $v_1 = b / \beta$.
2. For $k = 1, 2, \dots$:
   1. One Arnoldi step: it fills $v_{k+1}$ and column $k$ of $\bar H$. *(1 matvec.)*
   2. Solve $\min_y \|\beta e_1 - \bar H_k\, y\|_2$.
   3. Its residual norm is $\|r_k\|_2$. Stop if that is at most $\text{rtol} \cdot \|b\|_2$, or after a breakdown.
3. Form $x_k = V_k\, y$, once.

The stopping test needs only the residual norm, so the iterate is formed after the loop. Nothing inside the loop touches a length-$n$ vector except the Arnoldi step itself. The loop does still solve the whole small problem from scratch at every step, just to read off that norm. Step 4 removes that.

In [ ]:
def gmres_v2(matvec, b, *, rtol=1e-10, max_iterations=None, verbose=False):
    """GMRES through the Arnoldi relation: least squares on the small problem with H-bar.

    Each step solves min_y ||beta e_1 - H-bar_k y||, a (k+1) x k problem, and
    forms x_k = V_k y only once, at the end. Same iterates as gmres_v1.

    Args:
        matvec: function v -> A v. The solver never sees A itself.
        b: right-hand side, shape (n,).
        rtol: stop once ||r_k|| <= rtol * ||b||.
        max_iterations: largest k to try. Defaults to n.
        verbose: print y and the residual at every step.

    Returns:
        x: the last iterate x_k.
        record: dict with
            "iterations"        number of steps k taken,
            "matvecs"           number of matvec calls,
            "residual_history"  [||r_0||, ||r_1||, ..., ||r_k||],
            "converged"         whether the tolerance was reached.
    """
    n = b.shape[0]
    if max_iterations is None:
        max_iterations = n
    b_norm = np.linalg.norm(b)
    beta = b_norm

    # ----- k = 0: start from x_0 = 0, so the residual is b and v_1 = b / beta. -----
    residual_history = [b_norm]
    matvec_count = 0
    converged = False

    m = max_iterations
    V = np.zeros((n, m + 1))  # columns v_1, ..., v_{m+1}, filled one per step
    H = np.zeros((m + 1, m))  # H-bar, filled one column per step
    V[:, 0] = b / beta

    for k in range(1, max_iterations + 1):
        # 1. One Arnoldi step: v_{k+1} and column k of H-bar. Its only matvec.
        H[:, k - 1], breakdown = arnoldi_step_classical_gram_schmidt_twice(matvec, V, k - 1)
        matvec_count += 1

        # 2. The small least-squares problem, (k+1) x k. No length-n vectors.
        H_bar_k = H[: k + 1, :k]
        beta_e1 = np.zeros(k + 1)
        beta_e1[0] = beta
        y, *_ = np.linalg.lstsq(H_bar_k, beta_e1, rcond=None)

        # 3. Its residual is ||b - A V_k y||, since V_{k+1} has orthonormal columns.
        residual_history.append(np.linalg.norm(beta_e1 - H_bar_k @ y))

        if verbose:
            print(f"k = {k}:  y = {y},  ||r_k|| = {residual_history[-1]:.4e}")

        # 4. Stop once the residual is small enough.
        if residual_history[-1] <= rtol * b_norm:
            converged = True
            break
        # 5. No new direction: the Krylov space has stopped growing (Step 5).
        if breakdown:
            break

    # The iterate is needed only now: x_k = V_k y.
    x = V[:, :k] @ y

    record = {
        "iterations": k,
        "matvecs": matvec_count,
        "residual_history": residual_history,
        "converged": converged,
    }
    return x, record

In [ ]:
x, record = gmres_v2(CountingMatvec(A_geo3), b_geo3, verbose=True)
print("x =", x, "   matvecs:", record["matvecs"])
print()

for name, A, b, ks in [("geo3", A_geo3, b_geo3, None),
                       ("diag_narrow", A_narrow, b_narrow, range(1, 12)),
                       ("diag_spread", A_spread, b_spread, None)]:
    diff = compare_versions(gmres_v1, gmres_v2, A, b, ks=ks)
    print(f"{name:12s} gmres_v1 vs gmres_v2, k = 1..{diff['k'][-1]}:  "
          f"largest x_difference {diff['x_difference'].max():.1e},  "
          f"largest residual_difference {diff['residual_difference'].max():.1e}")
    assert diff["matvecs_equal"].all()
    assert diff["x_difference"].max() < 1e-11 and diff["residual_difference"].max() < 1e-11

The verbose run on `geo3` prints the same $y$ and the same residuals as `gmres_v1` did in Step 2. On all three matrices the two versions agree to round-off, with the same matvec counts. The largest difference between iterates is $5.3 \times 10^{-13}$, on `diag_spread`.

`gmres_v2` never computes the residual vector $b - A x_k$. Its residual is the norm from the small problem. Figure 3.1 checks that against the true residual of each iterate, $\|b - A x_k\|_2$, which `compare_versions` measures using $A$ directly.

In [ ]:
n_spread = b_spread.shape[0]
b_norm_spread = np.linalg.norm(b_spread)
_, record_v1 = gmres_v1(CountingMatvec(A_spread), b_spread)
x_v2, record_v2 = gmres_v2(CountingMatvec(A_spread), b_spread)
diff_v1_v2 = compare_versions(gmres_v1, gmres_v2, A_spread, b_spread)

fig, ax = plt.subplots(figsize=(6.5, 4), constrained_layout=True)
plot_residuals(ax, record_v1["residual_history"], b_norm=b_norm_spread, color=SERIES_COLORS[0],
               label=r"gmres_v1: $\|b - A V_k y\|_2$", rtol=1e-10)
plot_residuals(ax, record_v2["residual_history"], b_norm=b_norm_spread, color=SERIES_COLORS[1],
               marker="none", linestyle="--", label=r"gmres_v2: $\|\beta e_1 - \bar H_k y\|_2$")
ax.semilogy(diff_v1_v2["k"], diff_v1_v2["true_residual_new"], color=SERIES_COLORS[2], linestyle="none",
            marker="x", markersize=8, markeredgewidth=2, label=r"gmres_v2's $x_k$: true $\|b - A x_k\|_2$")
ax.set_title("gmres_v1 and gmres_v2 on diag_spread (n = 20)")
ax.legend(frameon=False, loc="center left")
style_axes(ax, xticks=range(0, n_spread + 1, 2))
add_figure_caption(fig, "3.1", "On diag_spread, gmres_v1's residual, gmres_v2's small-problem residual and the true residual of gmres_v2's iterates coincide.")
plt.show()
print(f"k = 20:  gmres_v1 {record_v1['residual_history'][20] / b_norm_spread:.1e}    "
      f"gmres_v2 small problem {record_v2['residual_history'][20] / b_norm_spread:.1e}    "
      f"true {diff_v1_v2['true_residual_new'][19]:.1e}")

# How well conditioned is the small problem, compared with Step 1's basis K_k?
K_spread = krylov_matrix(CountingMatvec(A_spread), b_spread, n_spread)
cond_K = np.array([np.linalg.cond(K_spread[:, :k]) for k in range(1, n_spread + 1)])
V_spread, H_spread, _ = arnoldi(CountingMatvec(A_spread), b_spread, n_spread)
print(f"cond(A) = {np.linalg.cond(A_spread):.1e}")
for k in (5, 10, 20):
    print(f"k = {k:2d}:  cond(H-bar_k) = {np.linalg.cond(H_spread[: k + 1, :k]):.1e}    "
          f"cond(K_k) = {cond_K[k - 1]:.1e}")
print("h_21,20 =", H_spread[20, 19], "   (the last step is flagged as a breakdown)")

In Figure 3.1 the three curves lie on top of each other all the way down, to about $10^{-12}$ at $k = 20$: $2.8 \times 10^{-13}$ for `gmres_v1`, $6.9 \times 10^{-13}$ from the small problem, and $6.3 \times 10^{-13}$ true. So the residual from the small problem is the true residual, to round-off.

At $k = 20 = n$ the Arnoldi step is flagged as a breakdown, as `geo3`'s third step was in Step 2: the Krylov space is all of $\mathbb{R}^{20}$, and $h_{21,20} = 0$ exactly. The small problem then has an exact solution, but its residual is not printed as 0, because `gmres_v2` computes it by subtraction, $\|\beta e_1 - \bar H_k\, y\|_2$, which carries the round-off of $y$.

The small problem is also well conditioned. $\operatorname{cond}(\bar H_k)$ grows to $1.0 \times 10^4$ at $k = 20$, the condition number of $A$ itself, while $\operatorname{cond}(K_{20})$ was $3.8 \times 10^{91}$. At $k = n$ that is no coincidence. The basis then spans all of $\mathbb{R}^n$, so $\bar H$ is just $A$ written in an orthonormal basis, and changing to an orthonormal basis doesn't change a condition number.

### 3.3 A look inside $\bar H_k$

To see the Hessenberg shape we need a nonsymmetric matrix. `diag_narrow` and `diag_spread` are diagonal, hence symmetric, and section 3.4 shows that a symmetric matrix gives a tridiagonal $\bar H$, which hides the shape. So we bring in a new example, `convdiff_n`. It discretises

$$
-u''(s) + c\, u'(s) = f(s) \quad \text{on } [0, 1], \qquad u(0) = u(1) = 0,
$$

on $n$ interior grid points $s_i = i\, \Delta s$, with spacing $\Delta s = 1/(n+1)$. Replacing $u''(s_i)$ by $(u_{i-1} - 2u_i + u_{i+1}) / \Delta s^2$ and $u'(s_i)$ by $(u_{i+1} - u_{i-1}) / (2 \Delta s)$ gives a tridiagonal matrix with

$$
\frac{2}{\Delta s^2} \text{ on the diagonal}, \qquad
-\frac{1}{\Delta s^2} - \frac{c}{2 \Delta s} \text{ below it}, \qquad
-\frac{1}{\Delta s^2} + \frac{c}{2 \Delta s} \text{ above it} .
$$

The parameter $c$ is a velocity. With $c = 0$ the matrix is symmetric: it is the 1D heat-equation matrix. With $c \ne 0$ the entries below and above the diagonal differ, so it isn't symmetric. That's all we need from it here. Step 6 comes back to it, to see what a larger $c$ does to convergence. We take $n = 20$, $c = 20$ and $f = 1$, so $b = (1, \dots, 1)$.

Figure 3.2 shows $\bar H_k$ after $k = 2, 4, 8$ and $16$ Arnoldi steps. Each entry is colored by its size on a log scale, and blank cells are exact zeros.

In [ ]:
A_cd, b_cd = convdiff_n(20, c=20.0)
_, H_cd, _ = arnoldi(CountingMatvec(A_cd), b_cd, 16)

# One color scale for all four panels: each H-bar_k is the top-left corner of H-bar_16.
smallest = np.abs(H_cd[H_cd != 0]).min()
largest = np.abs(H_cd).max()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), constrained_layout=True,
                         gridspec_kw={"width_ratios": [1, 1, 1.4, 2.2]})
for ax, k in zip(axes, (2, 4, 8, 16)):
    plot_matrix(ax, H_cd[: k + 1, :k], log=True, vmin=smallest, vmax=largest,
                title=rf"$\bar H_{{{k}}}$  ({k + 1} × {k})", colorbar=(k == 16))
add_figure_caption(fig, "3.2", '$\\bar H_k$ for convdiff_n (c = 20) after 2, 4, 8 and 16 Arnoldi steps. Blank cells are exact zeros: the Hessenberg shape.')
plt.show()

print("entries below the subdiagonal that are not exactly zero:", np.count_nonzero(np.tril(H_cd, -2)))

Each new step adds one column on the right and one row at the bottom, and the earlier entries never change. Below the subdiagonal everything is exactly zero, because those entries are never written. Above it, everything is filled in: $A v_j$ has a component along every earlier $v_i$, so every column is full down to its subdiagonal.

That has a cost. Step $j$ takes $j$ dot products and $j$ vector updates to remove those components, and all $j$ basis vectors must be kept. So work and memory both grow with $k$. Step 7 deals with that by restarting.

Next (Figure 3.3), how well the Arnoldi relation holds in floating point on `diag_spread`, next to Step 2's measure of how orthonormal $V_k$ is. Both are measured in the matrix 2-norm $\|M\|_2$, the largest factor by which $M$ stretches any vector. The relation's error is divided by $\|A\|_2$ to make it relative.

In [ ]:
eps = np.finfo(float).eps
steps = np.arange(1, n_spread + 1)
A_norm = np.linalg.norm(A_spread, 2)
orthogonality_loss = [np.linalg.norm(np.eye(k) - V_spread[:, :k].T @ V_spread[:, :k], 2) for k in steps]
relation_error = [np.linalg.norm(A_spread @ V_spread[:, :k] - V_spread[:, : k + 1] @ H_spread[: k + 1, :k], 2) / A_norm
                  for k in steps]

for k in (5, 20):
    print(f"k = {k:2d}:  ||I - V_k^T V_k|| = {orthogonality_loss[k - 1]:.1e}    "
          f"cond(V_k) = {np.linalg.cond(V_spread[:, :k]):.10f}    cond(K_k) = {cond_K[k - 1]:.1e}")
print(f"largest relation error over k = 1..20: {max(relation_error):.1e}")

fig, ax = plt.subplots(figsize=(6.5, 4), constrained_layout=True)
ax.semilogy(steps, orthogonality_loss, color=SERIES_COLORS[0], linewidth=2, marker="o", markersize=5,
            label=r"$\|I - V_k^\mathsf{T} V_k\|_2$  (orthogonality)")
ax.semilogy(steps, relation_error, color=SERIES_COLORS[1], linewidth=2, marker="s", markersize=5,
            label=r"$\|A V_k - V_{k+1} \bar H_k\|_2 \,/\, \|A\|_2$  (Arnoldi relation)")
ax.axhline(eps, color=REFERENCE_COLOR, linewidth=1)
ax.text(n_spread, eps * 1.6, r"$\varepsilon_{\mathrm{machine}}$", color=REFERENCE_COLOR, fontsize=9, ha="right", va="bottom")
ax.set_ylim(1e-18, 1e-10)
ax.set_xlabel("step k")
ax.set_title("Arnoldi with classical Gram–Schmidt applied twice, diag_spread")
ax.legend(frameon=False, loc="upper left")
style_axes(ax, xticks=range(0, n_spread + 1, 2))
add_figure_caption(fig, "3.3", 'Arnoldi with classical Gram–Schmidt applied twice on diag_spread: both the Arnoldi relation and orthonormality hold to round-off.')
plt.show()

Both curves in Figure 3.3 stay flat.

- **The Arnoldi relation holds to about $10^{-16}$ at every step.** It's built into the construction: each $h_{ij}$ records exactly what was subtracted, so $A v_j = \sum_i h_{ij} v_i$ holds to round-off however the subtractions went.
- **Orthonormality holds to round-off too.** $\|I - V_k^\mathsf{T} V_k\|_2$ is $2.5 \times 10^{-16}$ at $k = 5$ and $3.7 \times 10^{-16}$ at $k = 20$. To ten digits $\operatorname{cond}(V_k)$ prints as 1, against $3.8 \times 10^{91}$ for $K_{20}$. That is what the second Gram–Schmidt pass buys (Step 2, section 2.4).

The small-problem residual of 3.2 relies on both: on the relation, which holds by construction, and on $V_{k+1}$ being orthonormal, which two passes keep. Step 4's residual estimate will rely on orthonormality in the same way.

**When $V_{k+1}$ is not orthonormal.** Step 2 found a matrix where one pass of Gram–Schmidt loses orthogonality completely, `diag_spread(20, decades=12)`, with eigenvalues from 1 to $10^{12}$. On it, the next cell builds $V_{21}$ and $\bar H_{20}$ with each of Step 2's three step functions, and for every $k$ solves the small problem $\min_y \|\beta e_1 - \bar H_k\, y\|_2$ as `gmres_v2` does. For each method it prints $\|I - V_k^\mathsf{T} V_k\|_2$, the small problem's residual and the true residual $\|b - A V_k y\|_2$ of the iterate $x_k = V_k y$, the last two divided by $\|b\|_2$. (We use $A$ to measure the true residual.)

For comparison it also solves `gmres_v1`'s problem, $\min_y \|b - A V_k\, y\|_2$, on the one-pass classical basis. That needs $A V_k$, which `arnoldi` doesn't keep, so it costs $k$ more matvecs here. Figure 3.4 plots the true residuals.

In [ ]:
n_12 = b_spread_12.shape[0]
b_norm_12 = np.linalg.norm(b_spread_12)
steps_12 = np.arange(1, n_12 + 1)
methods = {   # label -> (step function, color, marker), as in Step 2's Figure 2.1
    "classical Gram–Schmidt, one pass": (arnoldi_step_classical_gram_schmidt, SERIES_COLORS[1], "s"),
    "modified Gram–Schmidt": (arnoldi_step_modified_gram_schmidt, SERIES_COLORS[2], "^"),
    "classical Gram–Schmidt applied twice": (arnoldi_step_classical_gram_schmidt_twice, SERIES_COLORS[0], "o"),
}

results = {}   # label -> arrays over k = 1, ..., 20
for label, (step, _, _) in methods.items():
    V, H, beta = arnoldi(CountingMatvec(A_spread_12), b_spread_12, n_12, step=step)
    loss, small, true = [], [], []
    for k in steps_12:
        beta_e1 = np.zeros(k + 1)
        beta_e1[0] = beta
        y, *_ = np.linalg.lstsq(H[: k + 1, :k], beta_e1, rcond=None)   # gmres_v2's small problem
        loss.append(np.linalg.norm(np.eye(k) - V[:, :k].T @ V[:, :k], 2))
        small.append(np.linalg.norm(beta_e1 - H[: k + 1, :k] @ y) / b_norm_12)
        true.append(np.linalg.norm(b_spread_12 - A_spread_12 @ (V[:, :k] @ y)) / b_norm_12)  # checking only
    results[label] = {"V": V, "loss": np.array(loss), "small": np.array(small), "true": np.array(true)}

# gmres_v1's problem, min_y ||b - A V_k y||, on the one-pass classical basis.
# arnoldi doesn't keep the images A v_j, so they cost one more matvec per column here.
V_one_pass = results["classical Gram–Schmidt, one pass"]["V"]
matvec = CountingMatvec(A_spread_12)
AV_one_pass = np.column_stack([matvec(V_one_pass[:, j]) for j in range(n_12)])
big_problem_one_pass = []
for k in steps_12:
    y, *_ = np.linalg.lstsq(AV_one_pass[:, :k], b_spread_12, rcond=None)
    big_problem_one_pass.append(np.linalg.norm(b_spread_12 - AV_one_pass[:, :k] @ y) / b_norm_12)

shown = (16, 17, 18, 19, 20)
print(" " * 42 + "".join(f"k = {k:<7d}" for k in shown))
for label, result in results.items():
    print(label)
    for name, key in (("||I - V_k^T V_k||_2", "loss"), ("small-problem residual", "small"), ("true residual", "true")):
        print(f"    {name:38s}" + "".join(f"{result[key][k - 1]:<11.1e}" for k in shown))
print("gmres_v1's problem on the one-pass classical basis")
print(f"    {'residual ||b - A V_k y||_2':38s}" + "".join(f"{big_problem_one_pass[k - 1]:<11.1e}" for k in shown))

x_exact_12 = np.linalg.solve(A_spread_12, b_spread_12)   # checking only
print(f"\nround-off in A x:  eps * ||A|| * ||x|| / ||b|| = "
      f"{eps * np.linalg.norm(A_spread_12, 2) * np.linalg.norm(x_exact_12) / b_norm_12:.1e}")

fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
for label, (_, color, marker) in methods.items():
    ax.semilogy(steps_12, results[label]["true"], color=color, linewidth=2, marker=marker, markersize=5,
                label=f"{label}: small problem")
ax.semilogy(steps_12, big_problem_one_pass, color=SERIES_COLORS[1], linewidth=2, linestyle="--",
            label="classical Gram–Schmidt, one pass: gmres_v1's problem")
ax.set_xlabel("step k")
ax.set_ylabel(r"true residual $\|b - A x_k\|_2 \,/\, \|b\|_2$")
ax.set_title("diag_spread(20, decades=12)")
ax.legend(frameon=False, loc="lower left")
style_axes(ax, xticks=range(0, n_12 + 1, 2))
add_figure_caption(fig, "3.4", "True residual of the small problem's iterate, with each of Step 2's three Arnoldi steps on diag_spread(20, decades=12). Dashed: gmres_v1's problem on the one-pass classical basis.")
plt.show()

Up to $k = 18$ all three methods give the same residuals to the digits printed, even though the one-pass classical basis has by then lost orthogonality to $1.7 \times 10^{-3}$. At $k = 19$ one-pass classical Gram–Schmidt parts from the other two. Its small problem reports $0.18$ and then $0.15$, while the true residual of its iterate stays at $0.24$ and $0.25$. The other two fall to $0.15$ at $k = 19$ and to about $10^{-5}$ at $k = 20$. That is where $V_{k+1}$ stops being orthonormal in any useful sense: at $k = 19$ the small problem uses $V_{20}$, and $\|I - V_{20}^\mathsf{T} V_{20}\|_2 = 1.1$.

The one-pass basis is still a good basis of the Krylov space, though. `gmres_v1`'s problem never assumes orthonormality, and with that same basis it reaches $4.9 \times 10^{-6}$ at $k = 20$ (dashed line in Figure 3.4). What fails is fact 2 of section 3.2: $\|V_{k+1} z\|_2 = \|z\|_2$ needs $V_{k+1}^\mathsf{T} V_{k+1} = I$. Without it, the small problem minimises the wrong quantity, and neither its $y$ nor its residual describes the true residual any more. That is also why losing orthogonality did no harm to `gmres_v1` in Step 2: it never relies on it.

The other two methods end at about $10^{-5}$, not at $10^{-13}$ as on the 4-decade `diag_spread`. That is not a failure of either. Computing $A x$ in floating point makes an error of about $\varepsilon_{\text{machine}} \|A\|_2 \|x\|_2$, which for this matrix is $5 \times 10^{-5}\, \|b\|_2$ (last line printed), so no method can promise a residual much below that, and both are within it. Modified Gram–Schmidt even ends slightly lower here, $8.7 \times 10^{-6}$ against $1.5 \times 10^{-5}$. What two passes buy is not a smaller residual. It is a basis orthonormal to $4 \times 10^{-16}$, so that the small problem means what it says, and, in Step 5, a breakdown test that works.

### 3.4 Aside: a symmetric $A$ gives a tridiagonal $\bar H$

Let $H_k$ be the top $k \times k$ block of $\bar H_k$, everything except its last row. Multiplying the Arnoldi relation on the left by $V_k^\mathsf{T}$, and using $V_k^\mathsf{T} V_k = I$ and $V_k^\mathsf{T} v_{k+1} = 0$, gives

$$
H_k = V_k^\mathsf{T} A V_k ,
$$

which is $A$ restricted to the Krylov space and written in the basis $V_k$. If $A$ is symmetric ($A^\mathsf{T} = A$), then so is $H_k$. A matrix that is both upper Hessenberg and symmetric is zero above its first superdiagonal as well as below its first subdiagonal: it is **tridiagonal**. So $A v_j$ only has components along $v_{j-1}$, $v_j$ and $v_{j+1}$, and each step needs just the last two basis vectors, at the same cost every step. This special case is the **Lanczos process**, the basis of MINRES and of the conjugate gradient method.

The Jacobian in the simulation isn't symmetric (it has advection and couples different fields), which is why it needs GMRES. This is also why `diag_narrow` and `diag_spread` would show a tridiagonal $\bar H$.

Figure 3.5 shows $\bar H_8$ for `convdiff_n` with $c = 0$ (symmetric) and $c = 20$ (not). On this plot, entries smaller than $10^{-12}$ times the largest are left blank, so that round-off doesn't fill in the zeros. We stop at $k = 8$ because, for $c = 0$ and $b = (1, \dots, 1)$, the Krylov space stops growing at $k = 10 = n/2$. Because $b$ is symmetric about the middle of the grid, it is a combination of only the 10 eigenvectors that share that symmetry, so $\mathcal{K}_k$ can never be bigger than 10-dimensional. That is a breakdown, and Step 5 is about it. With two passes the Arnoldi step catches it: asked for 12 steps, `arnoldi` stops after 10 (printed below).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
for ax, c in zip(axes, (0.0, 20.0)):
    A_c, b_c = convdiff_n(20, c=c)
    _, H_c, _ = arnoldi(CountingMatvec(A_c), b_c, 8)
    above_superdiagonal = np.abs(np.triu(H_c[:8, :8], 2)).max() / np.abs(H_c).max()
    print(f"c = {c:4.0f}:  largest |h_ij| above the superdiagonal, relative to the largest |h_ij|:"
          f" {above_superdiagonal:.1e}")
    kind = "symmetric" if c == 0 else "nonsymmetric"
    plot_matrix(ax, H_c, log=True, floor=1e-12, title=rf"$\bar H_8$ for convdiff_n, c = {c:.0f} ({kind})")
add_figure_caption(fig, "3.5", '$\\bar H_8$ is tridiagonal for the symmetric convdiff_n (c = 0), and full above the diagonal for the nonsymmetric one (c = 20).')
plt.show()

A_sym, b_sym = convdiff_n(20, c=0.0)
_, H_sym, _ = arnoldi(CountingMatvec(A_sym), b_sym, 12)
print(f"convdiff_n with c = 0, asked for 12 Arnoldi steps: stopped at a breakdown after {H_sym.shape[1]}")

With $c = 0$, the largest entry above the superdiagonal is $6.7 \times 10^{-17}$ times the largest entry, which is zero up to round-off: $\bar H_8$ is tridiagonal. With $c = 20$ that ratio is $1.7 \times 10^{-2}$, and the whole upper triangle is filled in.

### Summary of Step 3

- The numbers $h_{ij}$ that Arnoldi subtracts record each matvec in the new basis: $A v_j = \sum_{i=1}^{j+1} h_{ij}\, v_i$. Side by side, these equations are the **Arnoldi relation** $A V_k = V_{k+1} \bar H_k$, and $\bar H_k$ is upper Hessenberg.
- Because $b = \beta v_1$ and $V_{k+1}$ preserves lengths, $\|b - A V_k\, y\|_2 = \|\beta e_1 - \bar H_k\, y\|_2$.
- **`gmres_v2`** solves that $(k+1) \times k$ problem instead and forms $x_k$ once, at the end. Its iterates match `gmres_v1`'s to round-off, and its small problem is about as well conditioned as $A$ itself.
- In floating point the Arnoldi relation holds to round-off by construction. The equality of the two residuals also needs $V_{k+1}$ to be orthonormal. With classical Gram–Schmidt applied twice it is, to about $4 \times 10^{-16}$. With one pass it isn't: on the 12-decade `diag_spread` the small problem's iterate stalls at a residual of $0.25$, while `gmres_v1`'s problem on the same basis still reaches $5 \times 10^{-6}$ (3.3).
- Each step costs more than the last, because the columns of $\bar H_k$ keep getting longer. For a symmetric $A$, $\bar H_k$ is tridiagonal and every step costs the same (the Lanczos process).

**Ties to code.** In [gmres.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/gmres.py), `_arnoldi_steps` is the loop of `gmres_v2`. It calls `arnoldi_step` once per step, on a basis allocated once with `restart + 1` rows of zeros (`empty_basis` in [vector_ops.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/vector_ops.py)), the same idea as `V` here. Production never forms $A V_k$ and never calls a least-squares solver. It solves the small problem through [givens.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/givens.py), which is Step 4.

**Next: Step 4.** `gmres_v2` re-solves the whole small problem at every step only to learn the residual norm, even though $\bar H_k$ is just $\bar H_{k-1}$ with one column added. Step 4 updates a QR factorisation of $\bar H_k$ one column at a time, using Givens rotations, and the residual norm falls out of the update for free.